In [ ]:
!pip install datasets
#import torchquantum
!git clone https://github.com/mit-han-lab/torchquantum.git
!cd torchquantum && pip install --editable .
!git clone https://github.com/jemisjoky/TorchMPS.git
!cd TorchMPS && pip install .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
      Successfully uninstalled fsspec-2024.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.3.1+cu121 req

You need to restart jupyter notebook after this

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import torch.nn as nn
import torch.optim as optim
from transformers import GPT2Tokenizer, GPT2LMHeadModel, GPT2Model, AdamW, get_linear_schedule_with_warmup
from datasets import load_dataset
from torch.utils.data import DataLoader
import numpy as np
from torch.utils.checkpoint import checkpoint
import random
from torchmps import MPS
import time
import torchquantum as tq
import collections
import torch.optim as optim
import collections
import torch.nn.functional as F
from torch.distributions import Categorical
import math
import numpy as np


ImportError: cannot import name 'AdamW' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import torch.nn as nn
import torch.optim as optim
import collections
import torch.nn.functional as F
from torch.distributions import Categorical
import pandas as pd
import warnings
import numpy as np

In [ ]:
# Our search space will consist fo 64, 128, 256, 512 and sigamoid, tanh relu, idenitity.

class RNNSearchSpace():
  def __init__(self, ):
    super(RNNSearchSpace, self).__init__()
    self.target_classes = 32
    self.vocab = self.vocab_dict()

  def vocab_dict(self):
    # define all the allowed nodes and activation funcs
    nodes = [32,64, 128, 256]
    act_funcs = ['tanh', 'relu', 'elu', 'identity']

    # initialize list for keys and values for the vocab
    layer_params = []
    layer_id = []

    # go through the activation function for each node
    for i in range(len(nodes)):
      for j in range(len(act_funcs)):

        # create and id and a configuration tuple
        layer_params.append((nodes[i], act_funcs[j]))
        layer_id.append(len(act_funcs)*i+j)

    # zip the id and configurations into a dictionary
    vocab = dict(zip(layer_id, layer_params))
    # add dropout in the vocabulary
    vocab[len(vocab)+1] = (('dropout'))

    if self.target_classes == 2 :
      vocab[len(vocab)+1] = (self.target_classes - 1, 'sigmoid')
    else:

      # run another experiment where this is identity separetely
      vocab[len(vocab)+1] = (self.target_classes, 'softmax')
    #print(vocab)
    return vocab

  def encode_sequence(self, sequence):
    keys = list(self.vocab.keys())
    values = list(self.vocab.values())
    encoded_sequence = []
    for value in sequence:
      encoded_sequence.append(keys[values.index(value)])
    return encoded_sequence

  def decode_sequence(self, sequence):
    keys = list(self.vocab.keys())
    values = list(self.vocab.values())
    decoded_sequence = []
    for key in sequence:
      decoded_sequence.append(values[keys.index(key)])
    return decoded_sequence





In [ ]:
class Controller(RNNSearchSpace, nn.Module):
  def __init__(self):
    super().__init__( )
    self.max_len = 3
    self.controller_lstm_dim = 100
    self.controller_optimizer = "adam"
    self.controller_lr = 0.001
    self.controller_decay = 0.1
    self.controller_momentum = 0.0
    self.use_predictor = False
    self.target_classes = 32
    self.controller_loss_alpha = .9
    self.controller_weights = 'LOGS/controller_weights_1.pth'
    os.makedirs(os.path.dirname(self.controller_weights), exist_ok=True)
    self.controller_baseline_dec = 1
    self.baseline = torch.tensor(110, requires_grad=False)
    self.sampled_sequence_data = []
    self.counter = [0]*30
    self.stored_sequences = {}
    self.controller_classes = len(self.vocab)
    self.lstm = nn.LSTM(self.max_len-1, self.controller_lstm_dim, batch_first=True)
    self.dense = nn.Linear(self.controller_lstm_dim, self.controller_classes-2) # we do minus 2 we wont use dropout or softmax sampler
    self.softmax = nn.Softmax(dim=1)
    self.all_perplexities = []
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  def forward(self, x):
    x, _ = self.lstm(x)
    x = self.dense(x)
    x = self.softmax(x)
    return x

  def reset_sample(self):
    def reset(m):
      if hasattr(m, 'reset_parameters'):
        m.reset_parameters()
    self.apply(reset)
    self.sampled_sequence_data = []
    self.counter = [0]*30

  def add_model_metrics(self, sequences, perplexities, log_probs ):
    if len(perplexities) != len(sequences) != len(log_probs) :
      raise ValueError("Perplexities and Losses must be the same length")
    if len(perplexities) == 0 :
      raise ValueError("No Perplexities")
    for i in range(len(sequences)):
      self.stored_sequences[tuple(sequences[i])] = (perplexities[i], log_probs[i])
      self.all_perplexities.append(perplexities[i])

  def to_categorical(y, num_classes):
    """ 1-hot encodes a tensor """
    return np.eye(num_classes, dtype='uint8')[y]

  def custom_loss(self, perplexity, log_probs):
    log_probs = torch.stack(log_probs)
    optimizer = optim.Adam(self.parameters(), lr = .001, weight_decay = self.controller_decay)
    with torch.no_grad():
      self.baseline = self.baseline-self.controller_baseline_dec if self.baseline > 80 else self.baseline
    print
    data_array = np.array(self.all_perplexities)
    data_array = [self.baseline-i for i in data_array]
    mean = np.mean(data_array)
    std_dev = np.std(data_array)+1e-10
    reward = self.baseline-perplexity
    reward = reward/40
    x = log_probs * (reward)
    custom_loss = torch.sum(log_probs * (reward))/4
    optimizer.zero_grad()
    custom_loss.backward()
    optimizer.step()

  def one_hot_encode(self, tensor):
    tensor = tensor
    one_hot = torch.zeros(tensor.size(0), self.controller_classes)
    one_hot.scatter_(1, tensor.unsqueeze(1), 1)
    return one_hot

  def print_value(self):
    print(self.counter)

  def prepare_controller_data(self,sequence):
    perplexities = [self.stored_sequences[tuple(seq)][0] for seq in sequence]
    probs = [self.stored_sequences[tuple(seq)][1] for seq in sequence]
    return  perplexities, probs

  def train_control_model(self, sequence, epochs):
    perplexities, probs = self.prepare_controller_data(sequence)
    self.train()
    for epoch in range(epochs):
      for i in range(0, len(perplexities)):
        self.custom_loss(perplexities[i], probs[i])
    torch.save(self.state_dict(), self.controller_weights)

  def pad_sequence(self, sequence, max_len):
    padded_sequence = np.zeros((1, max_len ), dtype=int)
    padded_sequence[0, :len(sequence)] = sequence
    return padded_sequence

  def sample_architecture_sequences(self, number_of_samples):
    final_layer_id = len(self.vocab)
    dropout_id = final_layer_id -1 # the dropout is encoded to 27 which is the last id, if you run test code for sample search you can see
    vocab_idx = [0] + list(self.vocab.keys())
    samples = []
    log_probs = []
    while len(samples) < number_of_samples:
      seed = []
      sample_log_probs = []  # this are the probs for each step required for Reinforcement Learning algorithm
      while len(seed)< self.max_len:
        input_sequence = self.pad_sequence(seed, self.max_len-1)
        input_sequence = input_sequence.reshape(1, self.max_len-1)
        input_sequence = torch.tensor(input_sequence, dtype=torch.float32)
        logits = self.forward(input_sequence)
        logits = logits.view(-1)
        next_element = torch.multinomial(logits,1)
        next_element = next_element.view(1).item()
        logits = logits.view(1,final_layer_id-2)
        log_prob = F.cross_entropy(logits, torch.tensor([next_element]), reduction='none')

        if next_element == final_layer_id :
          seed.append(next_element)
          break
        elif len(seed) == self.max_len -1 :
          #seed.append(final_layer_id) #we are not going to use softmax makes it worse
          break
        else  :
          sample_log_probs.append(log_prob)
          seed.append(next_element)

      if seed not in self.sampled_sequence_data:
        samples.append(seed)
        log_probs.append(sample_log_probs)
        self.sampled_sequence_data.append(seed)
        for i in range(0, len(seed)):
          self.counter[seed[i]]+=1

    return samples, log_probs

In [ ]:
n_sub_hypernetwork = 1
epochs_qt = 1
epochs_classical = 0
batch_size_train = 1
batch_size_test  = 32

In [ ]:
def generate_qubit_states_torch(n_qubit, num_vectors):
    # Calculate the total number of possible combinations
    total_combinations = 2 ** n_qubit
    if num_vectors > total_combinations:
        raise ValueError(f"Number of vectors requested ({num_vectors}) exceeds the total number of unique combinations ({total_combinations}).")

    # Generate random unique indices
    random_indices = random.sample(range(total_combinations), num_vectors)
    random_indices = torch.tensor(random_indices, dtype=torch.int64).unsqueeze(1)

    # Convert indices to binary representation and then to -1 and 1
    qubit_states = ((random_indices.unsqueeze(2) & (1 << torch.arange(n_qubit, dtype=torch.int64))) > 0).float() * 2 - 1

    return qubit_states

In [ ]:
class MappingModel(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()
        # Initialize layers: an input layer, multiple hidden layers, and an output layer
        self.input_layer = nn.Linear(input_size, hidden_sizes[0])
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_sizes[i], hidden_sizes[i+1]) for i in range(len(hidden_sizes)-1)])
        self.output_layer = nn.Linear(hidden_sizes[-1], output_size)

    def forward(self, X):
        # Ensure the input tensor is the same type as the weights
        X = X.type_as(self.input_layer.weight)

        # Input layer with ReLU activation
        X = self.input_layer(X)

        # Hidden layers with ReLU activation
        for hidden in self.hidden_layers:
            X = hidden(X)

        # Output layer with linear activation
        output = self.output_layer(X)
        # output = F.tanh(output)  # It's often better to use ReLU or similar; tanh is used here as it was in the original model.
        return output


In [ ]:
class QLayer(nn.Module):
    def __init__(self, n_blocks, n_qubits_):
        super().__init__()

        self.n_wires = n_qubits_
        self.n_blocks = n_blocks
        self.ry_layers = tq.QuantumModuleList()
        self.cnot_layers = tq.QuantumModuleList()

        for _ in range(self.n_blocks):
            self.ry_layers.append(
                tq.Op1QAllLayer(
                    op=tq.RY,
                    n_wires=self.n_wires,
                    has_params=True,
                    trainable=True,
                )
            )
            self.cnot_layers.append(
                tq.Op2QAllLayer(
                    op=tq.CNOT,
                    n_wires=self.n_wires,
                    has_params=False,
                    trainable=False,
                    circular=False,
                )
            )

    def forward(self):
        qdev = tq.QuantumDevice(n_wires=self.n_wires, bsz=1, device=next(self.parameters()).device)
        device=next(self.parameters()).device
        easy_scale_coeff = 2**(self.n_wires-1)
        gamma = 0.1
        beta  = 0.8
        alpha = 0.3
        for k in range(self.n_blocks):
            self.ry_layers[k](qdev)
            self.cnot_layers[k](qdev)

        state_mag = qdev.get_states_1d().abs()[0]
        # state_mag = state_mag[:len(nw_list_normal)]
        x = torch.abs(state_mag) ** 2
        # x = torch.log(x)
        x = x.reshape(2**(self.n_wires),1)
        x = (beta*torch.tanh(gamma*easy_scale_coeff*x))**(alpha)
        x = x - torch.mean(x)
        x.to(device)
        return x

In [ ]:
class MultiLayerAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers):
        super(MultiLayerAttention, self).__init__()
        self.layers = nn.ModuleList([nn.MultiheadAttention(embed_dim, num_heads) for _ in range(num_layers)])

    def forward(self, x):
        attn_output = x
        for layer in self.layers:
            attn_output, _ = layer(attn_output, attn_output, attn_output)
        return attn_output

In [ ]:
class QT_HyperNet(nn.Module):
    def __init__(self, vocab_size, hidden_size, n_sub_hypernetwork, sampled_architecture, bond_dim = 32):
        super(QT_HyperNet, self).__init__()

        self.n_sub_hypernetwork = n_sub_hypernetwork
        self.weight_length = int(np.ceil((vocab_size * hidden_size) / self.n_sub_hypernetwork ))

        # self.n_qubit = int(np.ceil(np.log2(vocab_size * hidden_size))) # int(np.ceil(np.log2(vocab_size * hidden_size)))
        self.out_dim_MPS = 64 #256
        self.hidden_dim_MLP = 64
        self.out_dim_MLP = 20000
        self.batch_size = int(np.ceil((self.weight_length/self.out_dim_MLP))) #1000 #400 #4000

        # typically self.batch_size * self.out_dim_MLP * n_sub_hypernetwork should > vocab_size * hidden_size

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.init_mapping = "MLP"
        self.classical_layers = "MLP"

        self.n_qubit_qt = int(np.ceil(np.log2(self.batch_size)))
        self.n_qubit = self.n_qubit_qt
        self.q_depth    = 5
        self.QuantumNN = QLayer(self.q_depth, self.n_qubit_qt).to(self.device)

        if self.init_mapping == "MPS":
            # self.MappingNetwork = MPS(
            #     input_dim=self.n_qubit,
            #     output_dim=self.out_dim_MPS,
            #     bond_dim=bond_dim,
            #     init_std=1e-2 #5e-3
            # )
            print("Currently Unavailable")

        elif self.init_mapping == "MLP":
            self.MappingNetwork = MappingModel(self.n_qubit+1, [32, 64, 32], self.out_dim_MPS)


        self.activation_layer = []
        for size, activation in sampled_architecture:
            if activation == 'elu':
                self.activation_layer.append(nn.ELU())
            elif activation == 'tanh':
                self.activation_layer.append(nn.Tanh())
            elif activation == 'relu':
                self.activation_layer.append(nn.ReLU())
            elif activation == 'identity':
                self.activation_layer.append(nn.Identity())
            else:
              raise ValueError("Not allowed activation ")
        #. (32, 'indentity')
        if self.classical_layers == "MLP":
            self.fc1 = nn.Linear(self.out_dim_MPS, sampled_architecture[0][0])
            self.fc2 = nn.RNN(sampled_architecture[0][0],64)
            self.fc3 = nn.RNN(64,64)
            self.fc4 = nn.Linear(64, self.out_dim_MLP)  # Generate weights for lm_head

        elif self.classical_layers == "ATT":
            # Adding a multilayered attention model to the hypernetwork
            self.attention = MultiLayerAttention(embed_dim=self.out_dim_MPS, num_heads=16, num_layers=3)
            self.fc_final = nn.Linear(self.out_dim_MPS, self.out_dim_MLP)  # Final layer to match output dimension


    def forward(self):

        compute_method = "checkpoint"
        probs_ = self.QuantumNN().flatten()
        probs_ = probs_[:self.batch_size]
        probs_ = probs_.reshape(self.batch_size, 1, 1)

        # num_batches = 1 + ((self.weight_length + self.batch_size - 1) // self.batch_size) // self.out_dim_MLP
        qubit_states_torch = generate_qubit_states_torch(self.n_qubit, self.batch_size)[:self.weight_length].to(self.device)
        # print("self.n_qubit", self.n_qubit)

        # print("probs_.shape = ", probs_.shape )
        # print("qubit_states_torch.shape = ", qubit_states_torch.shape)

        combined_data_torch = torch.cat((qubit_states_torch, probs_), dim=2)


        prob_val_post_processed_list = []
        if compute_method == "checkpoint":

            # print("-----------------")
            # print("basis batch ", i, "/",num_batches)

            batch_data = combined_data_torch[0:self.batch_size]
            batch_data.requires_grad_()

            prob_val_post_processed_batch = checkpoint(self.MappingNetwork, batch_data)


            if self.classical_layers == "MLP":
                prob_val_post_processed_batch = checkpoint(self.fc1, prob_val_post_processed_batch)
                prob_val_post_processed_batch, _ = checkpoint(self.fc2, prob_val_post_processed_batch)
                prob_val_post_processed_batch = checkpoint(self.activation_layer[0], prob_val_post_processed_batch)
                prob_val_post_processed_batch, _  = checkpoint(self.fc3, prob_val_post_processed_batch)
                prob_val_post_processed_batch = checkpoint(self.activation_layer[1], prob_val_post_processed_batch)
                prob_val_post_processed_batch = checkpoint(self.fc4, prob_val_post_processed_batch)

            elif self.classical_layers == "ATT":
                prob_val_post_processed_batch = checkpoint(self.attention, prob_val_post_processed_batch)
                prob_val_post_processed_batch = checkpoint(self.fc_final, prob_val_post_processed_batch)


            prob_val_post_processed_list.append(prob_val_post_processed_batch)
            # print("prob_val_post_processed_batch.shape = ", prob_val_post_processed_batch.shape)

            torch.cuda.empty_cache()

        prob_val_post_processed_list = prob_val_post_processed_list[:self.weight_length]
        prob_val_post_processed = torch.cat(prob_val_post_processed_list, dim=0)

        # print("self.weight_length =", self.weight_length)
        # print("prob_val_post_processed.view(-1).shape = ", prob_val_post_processed.view(-1).shape)
        prob_val_post_processed = prob_val_post_processed.view(-1)[:self.weight_length]
        prob_val_post_processed = prob_val_post_processed - prob_val_post_processed.mean()

        torch.cuda.empty_cache()

        return prob_val_post_processed


In [ ]:
class QuantumTrainGPT2(nn.Module):
    def __init__(self, gpt2_model, vocab_size, hidden_size, n_sub_hypernetwork, sampled_architectures, pretrained_weights=False):
        super(QuantumTrainGPT2, self).__init__()
        self.gpt2_model = gpt2_model
        self.gpt2_model = self.gpt2_model.cuda()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.dtype = torch.float32  # Ensure all tensors are of this type

        self.sampled_architectures = sampled_architectures
        # Freeze all parameters in the gpt2_model
        for param in self.gpt2_model.parameters():
            param.requires_grad = False
        if pretrained_weights is True :
            module_list = []
            for i in range(n_sub_hypernetwork):
                model = QT_HyperNet(vocab_size, hidden_size, n_sub_hypernetwork, sampled_architectures)
                state_dict = torch.load(f'sub_hypernetwork_{sampled_architectures[0][0]}_{sampled_architectures[1][0]}_{i}.pth')
                model.load_state_dict(state_dict)
                module_list.append(model)
            self.grand_hypernetwork = nn.ModuleList(module_list).cuda()

        else:
            self.grand_hypernetwork = nn.ModuleList([
                QT_HyperNet(vocab_size, hidden_size, n_sub_hypernetwork, sampled_architectures)
                for _ in range(n_sub_hypernetwork)]).cuda()


    def save_model(self):
        for i, sub_hypernetwork in enumerate(self.grand_hypernetwork):
            state_dict = sub_hypernetwork.state_dict()
            torch.save(state_dict, f'sub_hypernetwork_{self.sampled_architectures[0][0]}_{self.sampled_architectures[1][0]}_{i}.pth')

    def forward(self, input_ids, attention_mask, labels):
        gen_weights = []
        for sub_hypernetwork in self.grand_hypernetwork:
            gen_weights.append(sub_hypernetwork())
        self.generated_weights = torch.cat(gen_weights, dim=0).view(-1)[:self.vocab_size * self.hidden_size].cuda()
        self.generated_weights = self.generated_weights.view(self.vocab_size, self.hidden_size).type(self.dtype)

        # outputs = self.gpt2_model(input_ids, attention_mask=attention_mask, labels=labels)
        outputs = self.gpt2_model(input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state.cuda().type(self.dtype)
        logits = F.linear(hidden_states, self.generated_weights, None)
        # Compute loss if labels are provided
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous().cuda()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)).cuda()
        return logits if loss is None else (loss, logits)
        # return loss

In [ ]:
class GPT2(nn.Module):
    def __init__(self, gpt2_model,  pretrained_lm_head_weights=None):
        super(GPT2, self).__init__()
        self.gpt2_model = gpt2_model
        self.lm_head = torch.nn.Linear(gpt2_model.config.n_embd, gpt2_model.config.vocab_size, bias=False)

        # Freeze all parameters in the gpt2_model
        for param in self.gpt2_model.parameters():
            param.requires_grad = False

        # Initialize lm_head with pretrained weights if provided
        if pretrained_lm_head_weights is not None:
            self.lm_head.weight.data = pretrained_lm_head_weights

    def forward(self, input_ids, attention_mask, labels):

        outputs = self.gpt2_model(input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state.cuda()
        logits = self.lm_head(hidden_states)

        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous().cuda()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)).cuda()

        return logits if loss is None else (loss, logits)


In [ ]:
# Initialize GPT-2 and Hypernetwork
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# Add padding token
tokenizer.pad_token = tokenizer.eos_token

# Load the base GPT-2 model without the language modeling head
gpt2_model = GPT2Model.from_pretrained('gpt2')

vocab_size = gpt2_model.config.vocab_size
hidden_size = gpt2_model.config.n_embd

print("vocab_size * hidden_size: ", vocab_size * hidden_size)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=512)

dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

vocab_size * hidden_size:  38597376


Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

In [ ]:
# Prepare the test dataset
test_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

test_loader = DataLoader(tokenized_test_dataset, batch_size=batch_size_test)

# Evaluation function to calculate loss and perplexity
def evaluate_model(model):
    model.eval()
    num_batches = 0
    data_loader = test_loader
    total_loss = 0
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            since_batch = time.time()
            inputs = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = inputs.clone().to(device)
            # if i > 1 :
            #     break  # temporary for testing
            loss, logits = model(inputs, attention_mask=attention_mask, labels=labels)
            # loss = outputs.loss
            total_loss += loss.item()
            num_batches += 1
            if i %40 == 0 :
                print(f"Step [{i+1}/{len(data_loader)}], batch time: {time.time() - since_batch:.2f}, Loss: {loss.item()}")
    avg_loss = total_loss / num_batches
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

In [ ]:
# Prepare Dataset
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader = DataLoader(tokenized_dataset, batch_size=batch_size_train, shuffle=True)

def train_model(model, isPretrained):

  model.cuda()
  optimizer = AdamW(model.parameters(), 2e-5)
  total_steps = len(train_loader)
  scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
  # Use train loader  and only 1 epoch
  num_batches = 0
  total_loss = 0
  loss_list = []
  for i, batch in enumerate(train_loader):
    since_batch = time.time()
    inputs = batch['input_ids'].cuda()

    labels = inputs.clone()  # For language modeling, labels are the same as inputs
    attention_mask = batch['attention_mask'].to(device)
    # Assume inputs, attention_mask, and labels are already defined
    inputs = inputs.to(device)
    attention_mask = attention_mask.to(device)
    labels = labels.to(device)
    with torch.cuda.amp.autocast():  # Enable mixed precision
        loss, logits = model(inputs, attention_mask, labels)

    loss_list.append(loss.cpu().detach().numpy())

    loss.backward()
    optimizer.step()

    scheduler.step()
    optimizer.zero_grad()
    # if i > 1 :
    #   break  # temporary for testing
    if  isPretrained and  i > (len(train_loader)/10):
      break
    if i % 100 == 0:
        print(f"Step [{i+1}/{len(train_loader)}], batch time: {time.time() - since_batch:.2f}, Loss: {loss.item()}")

  avg_loss = sum(loss_list) / len(loss_list)
  perplexity = math.exp(avg_loss)
  return avg_loss, perplexity



In [ ]:
# Initialize GPT-2 and Hypernetwork
model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# Add padding token
tokenizer.pad_token = tokenizer.eos_token

# Load the base GPT-2 model without the language modeling head
gpt2_model = GPT2Model.from_pretrained('gpt2')

vocab_size = gpt2_model.config.vocab_size
hidden_size = gpt2_model.config.n_embd

print("vocab_size * hidden_size: ", vocab_size * hidden_size)

# Save the Fine-Tuned Model
dir = f'results/sample_search'
os.makedirs(dir, exist_ok=True)
# As testing Scenario we will try to test several architectures for RNN for 1 layer by brute forcing to see if your method does work
# Search space [32, 64,128] [(relu, tanh), batchnorm][softmax]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create an instance of the Controller class
controller = Controller()

# This code will only sample two layer sequence since we find more complexities is worse

# First start with looping and sampling 50 architectures
number_samples = 1
epochs = 1

rnn_search_space = RNNSearchSpace()
prev_sample_layers = []
# This will just sample architecture train, then evaluate and and train the controller
def train():

  min_perplexity = 300
  min_architecture = None
  controller = Controller()

  for i in range(0,10):
    sampled_architectures, probs = controller.sample_architecture_sequences(number_samples)
    decoded_architecture = rnn_search_space.decode_sequence(sampled_architectures[0])
    #print(prev_sample_layers)
    # TODO modify to be able two sample for 2 layer architectures make model
    if (decoded_architecture[0][0], decoded_architecture[1][0]) in prev_sample_layers:

      #print("resample")
      print("cry",decoded_architecture)
      qt_gpt2 = QuantumTrainGPT2(gpt2_model, vocab_size, hidden_size, n_sub_hypernetwork, decoded_architecture, True).cuda()
      avg_loss, pply = train_model(qt_gpt2, True)
    else:
      qt_gpt2 = QuantumTrainGPT2(gpt2_model, vocab_size, hidden_size, n_sub_hypernetwork, decoded_architecture, False).cuda()
      avg_loss, pply = train_model(qt_gpt2, False)
      qt_gpt2.save_model()
      prev_sample_layers.append((decoded_architecture[0][0], decoded_architecture[1][0]))

    print(decoded_architecture)

    # Now we evaluate
    test_loss, test_perplexity = evaluate_model(qt_gpt2)
    print(f"Test Loss: {test_loss}")
    print(f"Test Perplexity: {test_perplexity}")

    result = ', '.join(str(decoded_architecture))
    i =0# channge manually
    np.array(test_loss).dump(dir+f"eqt_{i}_{result}_test_loss.dat")
    np.array(test_perplexity).dump(dir+f"{i}_{result}_test_perplexity.dat")

    if test_perplexity > 500:
      controller.add_model_metrics( sampled_architectures, [500], probs)
    else:
      controller.add_model_metrics( sampled_architectures, [test_perplexity], probs)
    # Train the controller model
    controller.train_control_model(sampled_architectures , epochs )

    ## TODO find the min
    if test_perplexity < min_perplexity:
      min_perplexity = test_perplexity
      min_architecture = decoded_architecture

  print("min_perplexity", min_perplexity)
  print("min_architecture", min_architecture)





# Old COde
# size = [32, 64, 128]
# activation = ['identity', 'relu', 'tanh']
# architecture = []

# for j in range(len(size)):
#     for k in range(len(activation)):
#         architecture.append((size[j], activation[k]))
# print(architecture)

# prev_sample_layers = []

# for sampled_architecture in architecture:
#   # We train, if previously sampled we train on only 1/5 of an epoch
#   if sampled_architecture[0] in prev_sample_layers:
#     qt_gpt2 = QuantumTrainGPT2(gpt2_model, vocab_size, hidden_size, n_sub_hypernetwork, sampled_architecture, True).cuda()

#     avg_loss, pply = train_model(qt_gpt2, True)
#   else:
#     qt_gpt2 = QuantumTrainGPT2(gpt2_model, vocab_size, hidden_size, n_sub_hypernetwork, sampled_architecture, False).cuda()
#     avg_loss, pply = train_model(qt_gpt2, False)
#     qt_gpt2.save_model()
#     prev_sample_layers.append(sampled_architecture[0])

#   print(sampled_architecture)

#   # Now we evaluate
#   test_loss, test_perplexity = evaluate_model(qt_gpt2)
#   print(f"Test Loss: {test_loss}")
#   print(f"Test Perplexity: {test_perplexity}")









vocab_size * hidden_size:  38597376


In [ ]:
train()

Step [1/36718], batch time: 0.16, Loss: 99.59185791015625
Step [101/36718], batch time: 0.12, Loss: 48.25691223144531
Step [201/36718], batch time: 0.12, Loss: 9.040912628173828
Step [301/36718], batch time: 0.12, Loss: 7.576099395751953
Step [401/36718], batch time: 0.12, Loss: 4.757177829742432
Step [501/36718], batch time: 0.12, Loss: 18.678442001342773
Step [601/36718], batch time: 0.14, Loss: 5.499143600463867
Step [701/36718], batch time: 0.12, Loss: 9.80892276763916
Step [801/36718], batch time: 0.12, Loss: 4.4103474617004395
Step [901/36718], batch time: 0.13, Loss: 12.315557479858398
Step [1001/36718], batch time: 0.12, Loss: 3.937316417694092
Step [1101/36718], batch time: 0.13, Loss: 3.985712766647339
Step [1201/36718], batch time: 0.12, Loss: 4.177424907684326
Step [1301/36718], batch time: 0.12, Loss: 5.424015522003174
Step [1401/36718], batch time: 0.13, Loss: 10.169042587280273
Step [1501/36718], batch time: 0.12, Loss: 4.161807537078857
Step [1601/36718], batch time: 0.

In [ ]:
# Save the Fine-Tuned Model
directory = f'result/batch_{batch_size_train}/n_sub_{n_sub_hypernetwork}/'
os.makedirs(directory, exist_ok=True)

qt_gpt2.gpt2_model.save_pretrained(directory+f'n_sub_{n_sub_hypernetwork}_qtgpt2_eqt_{epochs_qt}_ecl_{epochs_classical}')
tokenizer.save_pretrained(directory+f'n_sub_{n_sub_hypernetwork}_qtgpt2_eqt_{epochs_qt}_ecl_{epochs_classical}')
np.array(loss_list).dump(directory+f"_eqt_{epochs_qt}_ecl_{epochs_classical}_loss_list.dat")


In [ ]:
# Calculate loss and perplexity on the test set
if epochs_classical == 0:
    test_model = qt_gpt2
elif epochs_classical > 0:
    test_model = gpt2
test_loss, test_perplexity = evaluate_model(test_model, test_loader)

print(f"Test Loss: {test_loss}")
print(f"Test Perplexity: {test_perplexity}")

np.array(test_loss).dump(directory+f"eqt_{epochs_qt}_ecl_{epochs_classical}_test_loss.dat")
np.array(test_perplexity).dump(directory+f"eqt_{epochs_qt}_ecl_{epochs_classical}_test_perplexity.dat")


In [ ]:
# Calculate loss and perplexity on the training set
train_loss, train_perplexity = evaluate_model(test_model, train_loader)
print(f"Train Loss: {train_loss}")
print(f"Train Perplexity: {train_perplexity}")

np.array(train_loss).dump(directory+f"eqt_{epochs_qt}_ecl_{epochs_classical}_train_loss.dat")
np.array(train_perplexity).dump(directory+f"eqt_{epochs_qt}_ecl_{epochs_classical}_train_perplexity.dat")


In [ ]:
# Training Loop
qt_gpt2.cuda()


loss_list = []
for epoch in range(epochs_qt):
    for i, batch in enumerate(train_loader):

        since_batch = time.time()

        inputs = batch['input_ids'].cuda()
        labels = inputs.clone()  # For language modeling, labels are the same as inputs

        attention_mask = batch['attention_mask'].to(device)


        with torch.cuda.amp.autocast():  # Enable mixed precision
            loss, logits = qt_gpt2(inputs, attention_mask, labels)

        loss_list.append(loss.cpu().detach().numpy())
        loss.backward()
        optimizer.step()


        scheduler.step()
        optimizer.zero_grad()
        if i % 1 == 0:
            print(f"Epoch [{epoch + 1}/{epochs_qt+epochs_classical}], Step [{i+1}/{len(train_loader)}], batch time: {time.time() - since_batch:.2f}, Loss: {loss.item()}")


if epochs_qt == 0:
    gpt2 = GPT2(qt_gpt2.gpt2_model) # we don't have initial gpt2 parameters from qt in this case

elif epochs_qt > 0:
    gpt2 = GPT2(qt_gpt2.gpt2_model, qt_gpt2.generated_weights)

gpt2.cuda()
optimizer = AdamW(gpt2.parameters(), 2e-5)
total_steps = len(train_loader) * epochs_classical
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

for epoch in range(epochs_classical):
    for i, batch in enumerate(train_loader):

        since_batch = time.time()

        inputs = batch['input_ids'].to(device)
        labels = inputs.clone()  # For language modeling, labels are the same as inputs

        attention_mask = batch['attention_mask'].to(device)

        with torch.cuda.amp.autocast():  # Enable mixed precision
            loss, logits = gpt2(inputs, attention_mask, labels)

        loss_list.append(loss.cpu().detach().numpy())
        loss.backward()
        optimizer.step()

        scheduler.step()
        optimizer.zero_grad()
        if i % 100 == 0:
            print(f"Epoch [{epoch + epochs_qt + 1}/{epochs_qt+epochs_classical}], Step [{i+1}/{len(train_loader)}], batch time: {time.time() - since_batch:.2f}, Loss: {loss.item()}")